<div id="singlestore-header" style="display: flex; background-color: rgba(235, 249, 245, 0.25); padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/browser.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px;">Build and Deploy a Custom MCP Server on SingleStore Cloud Functions</h1>
    </div>
</div>

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Note</b></p>
        <p>This notebook can be run on a Free Starter Workspace. To create a Free Starter Workspace navigate to <tt>Start</tt> using the left nav. You can also use your existing Standard or Premium workspace with this Notebook.</p>
    </div>
</div>

In [1]:
%%sql
DROP DATABASE IF EXISTS mcp_demo;
CREATE DATABASE mcp_demo;

USE mcp_demo;

CREATE TABLE products (
    id INT PRIMARY KEY,
    name VARCHAR(255),
    category VARCHAR(100),
    price DECIMAL(10,2),
    stock INT,
    description TEXT
);

INSERT INTO products VALUES
(1, 'SingleStore T-Shirt', 'Apparel', 29.99, 150, 'Comfortable cotton tee with the SingleStore logo'),
(2, 'Developer Hoodie', 'Apparel', 59.99, 75, 'Warm hoodie for late-night coding sessions'),
(3, 'Mechanical Keyboard', 'Electronics', 149.99, 30, 'Cherry MX Blue switches, RGB backlight'),
(4, 'Ergonomic Mouse', 'Electronics', 79.99, 50, 'Vertical design reduces wrist strain'),
(5, 'Standing Desk', 'Furniture', 499.99, 20, 'Electric height-adjustable desk'),
(6, 'Monitor Light Bar', 'Electronics', 39.99, 100, 'Reduce eye strain with screen-mounted lighting'),
(7, 'Cable Management Kit', 'Accessories', 19.99, 200, 'Keep your desk clean and organized'),
(8, 'Laptop Stand', 'Accessories', 34.99, 80, 'Aluminum, adjustable height and angle'),
(9, 'Noise Cancelling Headphones', 'Electronics', 299.99, 25, 'ANC with 30-hour battery life'),
(10, 'Webcam HD Pro', 'Electronics', 89.99, 60, '1080p with auto-focus and built-in mic');

CREATE TABLE orders (
    id INT PRIMARY KEY,
    product_id INT,
    customer_name VARCHAR(255),
    quantity INT,
    order_date DATE,
    status VARCHAR(50)
);

INSERT INTO orders VALUES
(1, 1, 'Alice', 2, '2025-04-01', 'shipped'),
(2, 3, 'Bob', 1, '2025-04-02', 'delivered'),
(3, 5, 'Charlie', 1, '2025-04-03', 'processing'),
(4, 2, 'Diana', 3, '2025-04-04', 'shipped'),
(5, 9, 'Eve', 1, '2025-04-05', 'delivered'),
(6, 4, 'Frank', 2, '2025-04-06', 'processing'),
(7, 6, 'Grace', 4, '2025-04-07', 'shipped'),
(8, 10, 'Hank', 1, '2025-04-08', 'delivered'),
(9, 7, 'Ivy', 5, '2025-04-09', 'shipped'),
(10, 8, 'Jack', 1, '2025-04-10', 'processing');

## Step 2: Install Dependencies

We use the [`fastmcp`](https://github.com/jlowin/fastmcp) library (FastMCP 2.x). It exposes a clean `http_app()` method that returns an ASGI app whose routes and lifespan can be composed directly into a FastAPI app — which is exactly what the SingleStore Cloud Function harness (`singlestoredb.apps.run_function_app`) expects.

**Why the explicit pins and the `sys.modules` purge below.** The Aura runtime pre-loads `anyio` at kernel startup (via `singlestoredb.apps`). When we `pip install fastmcp`, pip upgrades `anyio` on disk to a version that has `AnyByteStreamConnectable`, but `sys.modules` still holds the **old** `anyio.abc` from kernel startup. The next fresh `import anyio.streams.text` then does `from ..abc import AnyByteStreamConnectable`, finds the stale `anyio.abc` in `sys.modules`, and blows up with `ImportError: cannot import name 'AnyByteStreamConnectable' from 'anyio.abc'`. We also pin `starlette<0.47` so the preinstalled `fastapi 0.115.12` stops complaining (it requires `starlette<0.47.0,>=0.40.0`).

| Package | Constraint | Reason |
|---|---|---|
| `anyio` | `>=4.10` | `mcp` (dependency of `fastmcp`) imports `AnyByteStreamConnectable`, added in anyio **4.10**. |
| `starlette` | `<0.47.0,>=0.40.0` | The preinstalled FastAPI `0.115.12` requires this range; `fastmcp`/`mcp` want `starlette>=0.27` — both satisfied here. |
| `fastmcp` | (latest) | Provides `FastMCP.http_app()` and composable ASGI routes/lifespan. |

In [2]:
%pip install --quiet \
    "anyio>=4.10" \
    "starlette<0.47.0,>=0.40.0" \
    "fastmcp"

# Clear any stale imports from the runtime's default versions of these
# libraries so the freshly installed ones are picked up without a kernel
# restart. `anyio` in particular is loaded early by `singlestoredb.apps`,
# so its `abc` submodule is cached before pip upgrades the files on disk.
import sys
for _mod in list(sys.modules):
    if any(
        _mod == p or _mod.startswith(p + ".")
        for p in ("mcp", "anyio", "starlette", "fastmcp", "sniffio", "httpx")
    ):
        del sys.modules[_mod]


# Verify the versions we actually loaded.
from importlib.metadata import version as _pkg_version
import fastmcp, mcp, anyio, starlette  # force import; missing pkg fails loudly
for _pkg in ("fastmcp", "mcp", "anyio", "starlette"):
    print(f"{_pkg:<9} == {_pkg_version(_pkg)}")
print("Dependencies ready.")

> If an import in a later cell still fails (e.g. you re-ran Step 2 and cells in between), use **Kernel → Restart** and re-run the notebook from the top. That clears any C-extension state that `del sys.modules` cannot. The `pyopenssl`/`cryptography` warning is a pre-existing state of the Aura runtime and is safe to ignore.

## Step 3: Create the Database Connection

Set up a reusable SQLAlchemy engine against the `mcp_demo` database. The `connection_url` variable is automatically injected by the SingleStore Notebook runtime when a deployment is selected.

In [3]:
from sqlalchemy import create_engine, text
import requests
import logging
import json

logging.basicConfig(level=logging.INFO)

# Download the SingleStore CA certificate for SSL connections
ca_cert_url = "https://portal.singlestore.com/static/ca/singlestore_bundle.pem"
ca_cert_path = "/tmp/singlestore_bundle.pem"

response = requests.get(ca_cert_url)
with open(ca_cert_path, "wb") as f:
    f.write(response.content)

# Create the SQLAlchemy engine using the notebook's built-in connection_url.
# The built-in `connection_url` may or may not end with a trailing slash or
# include a default database name; strip both so we can reliably append our
# own database (`mcp_demo`).
sql_conn_string = connection_url.replace("singlestoredb", "mysql+pymysql").rstrip("/")
# If connection_url already points at a specific database (e.g. ".../defaultdb"),
# swap it for mcp_demo rather than nesting paths.
if "/" in sql_conn_string.split("://", 1)[-1]:
    sql_conn_string = sql_conn_string.rsplit("/", 1)[0]

engine = create_engine(
    f"{sql_conn_string}/mcp_demo?ssl_ca={ca_cert_path}",
    pool_size=5,
    max_overflow=3,
    pool_timeout=30,
)

def run_query(query: str) -> list[dict]:
    """Execute a SQL query and return results as a list of dicts."""
    with engine.connect() as conn:
        result = conn.execute(text(query))
        columns = list(result.keys())
        rows = result.fetchall()
        return [dict(zip(columns, row)) for row in rows]

logging.info("Database connection established.")

## Step 4: Build the MCP Server

Create a `FastMCP` instance and register tools with `@mcp.tool()`. Each tool's docstring and type hints are surfaced to the MCP client (Claude) as the tool's description and schema.

In [4]:
from fastmcp import FastMCP

mcp = FastMCP(name="SingleStore Product Catalog")


@mcp.tool()
def health() -> str:
    """
    Liveness check for the SingleStore MCP server. Returns 'ok' if the
    server is reachable and SQL connections are working.
    """
    try:
        run_query("SELECT 1 AS ok")
        return "ok"
    except Exception as e:
        return f"error: {e}"


@mcp.tool()
def search_products(query: str, category: str = "") -> str:
    """
    Search the product catalog by name or description.
    Optionally filter by category.

    Args:
        query: Search term to match against product name or description
        category: Optional category filter (e.g., 'Electronics', 'Apparel')
    """
    sql = """
        SELECT id, name, category, price, stock, description
        FROM products
        WHERE (name LIKE CONCAT('%', :query, '%') OR description LIKE CONCAT('%', :query, '%'))
    """
    params = {"query": query}
    if category:
        sql += " AND category = :category"
        params["category"] = category

    results = run_query(sql, params)
    if not results:
        return f"No products found matching '{query}'."
    return json.dumps(results, indent=2, default=str)


@mcp.tool()
def get_product_details(product_id: int) -> str:
    """
    Get full details for a specific product by its ID.

    Args:
        product_id: The numeric ID of the product
    """
    results = run_query("SELECT * FROM products WHERE id = :product_id", {"product_id": product_id})
    if not results:
        return f"No product found with ID {product_id}."
    return json.dumps(results[0], indent=2, default=str)


@mcp.tool()
def check_inventory(category: str = "") -> str:
    """
    Check current inventory levels. Optionally filter by category.
    Returns products sorted by stock level (lowest first).

    Args:
        category: Optional category filter
    """
    sql = "SELECT name, category, stock, price FROM products"
    params = {}
    if category:
        sql += " WHERE category = :category"
        params["category"] = category
    sql += " ORDER BY stock ASC"
    return json.dumps(run_query(sql, params), indent=2, default=str)


@mcp.tool()
def get_recent_orders(limit: int = 10) -> str:
    """
    Get the most recent orders with product details.

    Args:
        limit: Number of orders to return (default 10)
    """
    sql = """
        SELECT o.id AS order_id, p.name AS product,
               o.customer_name, o.quantity, o.order_date, o.status
        FROM orders o
        JOIN products p ON o.product_id = p.id
        ORDER BY o.order_date DESC
        LIMIT :limit
    """
    return json.dumps(run_query(sql, {"limit": limit}), indent=2, default=str)


@mcp.tool()
def get_sales_summary() -> str:
    """
    Get a summary of sales by category, including
    total revenue and units sold.
    """
    sql = """
        SELECT p.category,
               COUNT(o.id) AS total_orders,
               SUM(o.quantity) AS total_units,
               ROUND(SUM(o.quantity * p.price), 2) AS total_revenue
        FROM orders o
        JOIN products p ON o.product_id = p.id
        GROUP BY p.category
        ORDER BY total_revenue DESC
    """
    return json.dumps(run_query(sql), indent=2, default=str)


@mcp.tool()
def run_custom_query(sql_query: str) -> str:
    """
    Run a read-only SQL query against the mcp_demo database.
    Only SELECT statements are allowed.

    Args:
        sql_query: A SELECT SQL query to execute
    """
    if not sql_query.strip().upper().startswith("SELECT"):
        return "Error: Only SELECT queries are allowed."
    try:
        return json.dumps(run_query(sql_query), indent=2, default=str)
    except Exception as e:
        return f"Query error: {e}"


logging.info("MCP server created with tools registered.")

## Step 5: Wrap the MCP Server in FastAPI

Cloud Functions are served by FastAPI. `fastmcp` gives us an ASGI app via `mcp.http_app(path="/mcp")`; we pass its `routes` and `lifespan` directly into a new `FastAPI` instance — and **we deliberately do not add any additional `@app.get`/`@app.post` endpoints**.

> Why no extra FastAPI endpoints? The SingleStore Cloud Function gateway builds its route map from what FastAPI exposes via `openapi.json`. If the app declares **no** user-defined APIRoutes, the gateway passes every request through to uvicorn, so your fastmcp-registered `/mcp` route serves requests normally. The moment you add a single `@app.get("/health")`, the gateway flips to a filtering mode and only proxies paths that appear in `openapi.json` — which excludes plain Starlette `Route` objects (like the one fastmcp registers for `/mcp`). Any unknown path then gets a 307 slash-toggle fallback from the gateway, which also strips the `/functions/<id>` prefix and loops. To keep `/mcp` reachable, we keep the FastAPI app pristine — the MCP protocol itself handles tool discovery (`tools/list` JSON-RPC method) and liveness (just a plain `initialize` works).
>
> Do **not** use `app.mount("/mcp", ...)`: `Mount` entries are also dropped by the same gateway filtering.

In [5]:
from fastapi import FastAPI, Request
from fastapi.responses import Response

# Get fastmcp's ASGI handler — this is the real MCP protocol implementation
# that handles initialize, tools/list, tools/call, SSE streaming, etc.
mcp_asgi_app = mcp.http_app(path="/mcp")
_fastmcp_mcp_route = next(
    r for r in mcp_asgi_app.routes if getattr(r, "path", None) == "/mcp"
)
_mcp_asgi_handler = _fastmcp_mcp_route.app

app = FastAPI(
    title="SingleStore MCP Server",
    description="A custom MCP server deployed as a SingleStore Cloud Function",
    version="1.0.0",
    lifespan=mcp_asgi_app.lifespan,
)


class ASGIProxyResponse(Response):
    """A FastAPI/Starlette `Response` whose `__call__` delegates the entire
    ASGI interaction (headers, body, SSE streaming) to another ASGI app.

    Why: the SingleStore Cloud Function gateway only proxies requests to
    paths it sees in `openapi.json`. `fastmcp` registers `/mcp` as a plain
    Starlette `Route`, which FastAPI's schema generator never picks up —
    so the gateway 307-loops requests (`/mcp` ↔ `/mcp/`, prefix stripped).
    Adding an `@app.post("/mcp")` stub fixes the gateway's view but then
    the stub intercepts the request instead of `fastmcp`. Returning this
    proxy response from the stub hands the ASGI scope/receive/send off to
    `fastmcp` untouched, so SSE streaming and all MCP protocol semantics
    work end-to-end.

    We buffer the request body at handler time (via `request.body()`) and
    replay it on fastmcp's first `receive()` call. Subsequent `receive()`
    calls fall through to the real client connection so SSE keepalive and
    disconnect signals work normally.
    """

    def __init__(self, target, scope, body: bytes, real_receive):
        super().__init__(content=b"", status_code=200)
        self._target = target
        self._scope = scope
        self._body = body
        self._real_receive = real_receive

    async def __call__(self, scope, receive, send):
        replayed = False

        async def _receive():
            nonlocal replayed
            if not replayed:
                replayed = True
                return {"type": "http.request", "body": self._body, "more_body": False}
            return await self._real_receive()

        await self._target(self._scope, _receive, send)


@app.post("/mcp")
@app.get("/mcp", include_in_schema=False)
@app.delete("/mcp", include_in_schema=False)
async def mcp_endpoint(request: Request):
    """MCP Streamable HTTP endpoint — delegates to fastmcp's ASGI handler."""
    body = await request.body()
    return ASGIProxyResponse(
        _mcp_asgi_handler, request.scope, body, request.receive
    )

## Step 6: Start the Cloud Function Server

Run the FastAPI app using the SingleStore Cloud Function SDK. This gives you a temporary interactive URL to test against before publishing.

In [6]:
import singlestoredb.apps as apps

connection_info = await apps.run_function_app(app)

You'll see a temporary URL and a Swagger UI. Confirm `/health` returns `{"status":"ok",...}` before moving on.

## Step 7: Publish as a Cloud Function

Once the interactive run works:

1. Click **Publish** in the top-right corner of the notebook.
2. Select **Cloud Function**.
3. Give it a name, e.g. `mcp-product-catalog`.
4. Select your SingleStore deployment (workspace).
5. Choose a runtime size (Small is fine for testing).
6. Click **Publish**.

After a few moments the status changes to **Active**. You now have a stable HTTPS URL like:


```
https://apps.us-east-1.cloud.singlestore.com/functions/<function-id>/
```

The MCP endpoint is at:


```
https://apps.us-east-1.cloud.singlestore.com/functions/<function-id>/mcp
```

## Step 8: Create a Container App API Key

Your Cloud Function is protected by JWT authentication. To call it you need an API Key.

1. Navigate to **Container Services > Cloud Functions** in the Cloud Portal.
2. Select your `mcp-product-catalog` function.
3. Click **View API Keys** in the top-right.
4. Click **Create API Key**.
5. Enter a name (e.g., `claude-mcp-key`) and set an expiration date.
6. Click **Create API Key** and copy the key immediately — it is shown only once.

You'll use this key as a Bearer token: `Authorization: Bearer <YOUR_API_KEY>`.

## Step 9: Quick Test with curl

The deployed MCP server exposes a single path, `/mcp`, which implements the MCP streamable-HTTP transport. Per the [MCP spec](https://modelcontextprotocol.io/specification/2025-03-26/basic/transports#streamable-http) it accepts `POST` for JSON-RPC messages, `GET` to open a server-sent-events stream, and `DELETE` to close a session. Use `initialize` as the liveness check.


```shell
export BASE_URL="https://apps.us-east-1.cloud.singlestore.com/functions/<function-id>"
export MCP_URL="$BASE_URL/mcp"
export API_KEY="<your-api-key>"

# MCP initialize (POST) — should return the server's capabilities
# and an `mcp-session-id` response header.
curl -sS -i -X POST "$MCP_URL" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -H "Authorization: Bearer $API_KEY" \
  -d '{
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
      "protocolVersion": "2025-03-26",
      "capabilities": {},
      "clientInfo": {"name": "curl-test", "version": "1.0.0"}
    }
  }'
```

A successful response looks like:


```
HTTP/2 200
content-type: text/event-stream
mcp-session-id: <session-id>
...
event: message
data: {"jsonrpc":"2.0","id":1,"result":{"protocolVersion":"2025-03-26","capabilities":{...},"serverInfo":{"name":"SingleStore Product Catalog","version":"..."}}}
```

For a simple liveness check you can also call the `health` tool once initialized, or inspect tool listings via `tools/list`.

> **Troubleshooting**
> - **`307 Temporary Redirect` with `Location: http://apps.us-east-1.cloud.singlestore.com/mcp/` (prefix stripped):** the SingleStore gateway has put the function into "filtered" mode because an `@app.get`/`@app.post`/etc. endpoint was added in Step 5. Remove any such endpoints so the FastAPI app has **no** user-defined APIRoutes, rerun Steps 4–6, and republish.
> - **`401 Unauthorized`:** the Container App API Key has expired or is wrong. Generate a new one (Step 8) and re-export `API_KEY`.

## Step 10: Connect Claude Code

Claude Code supports remote MCP servers natively. Add your server with one command:


```shell
claude mcp add singlestore-catalog \
  --transport http \
  "$MCP_URL" \
  --header "Authorization: Bearer $API_KEY"
```

Start Claude Code and try prompts like:

- "Search for products related to keyboards."
- "What's in stock in the Electronics category?"
- "Show me the recent orders."
- "Give me a sales summary by category."

Claude discovers the tools from your MCP server automatically and calls them to answer your questions.

## Adding More Tools

Extending this is just adding another `@mcp.tool()` function and republishing.

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>